# 03: 量子テレポーテーション — 回路の実装

ノート `03_quantum_teleportation.md` の「プロトコルの回路表現」以降を Qiskit で実装し、シミュレーションで検証する。

**内容:**
1. ベル対の生成回路
2. 量子テレポーテーション回路（全プロトコル）
3. シミュレーションによる検証

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector
import numpy as np

![量子テレポーテーション回路](../images/03_teleportation.png)

## 1. ベル対の生成

$$
|00\rangle \xrightarrow{H \otimes I} \frac{|0\rangle + |1\rangle}{\sqrt{2}} |0\rangle \xrightarrow{\text{CNOT}} \frac{|00\rangle + |11\rangle}{\sqrt{2}} = |\Phi^+\rangle
$$

$q_0$ にアダマールゲート → $q_0$（制御）$q_1$（標的）の CNOT でベル状態を生成する。

In [ ]:
# ベル対生成回路
bell = QuantumCircuit(2, name="Bell pair")
bell.h(0)
bell.cx(0, 1)
bell.draw("mpl")

In [ ]:
# 状態ベクトルを確認: |00⟩ + |11⟩ / √2
sv_bell = Statevector.from_instruction(bell)
print("ベル状態 |Φ+⟩:")
print(sv_bell.draw("text"))
print()
print("各基底の振幅:")
for i, amp in enumerate(sv_bell):
    if abs(amp) > 1e-10:
        print(f"  |{i:02b}⟩: {amp:.4f}")

## 2. 量子テレポーテーション回路

回路は3つのフェーズからなる：

1. **準備フェーズ:** $q_1, q_2$ にベル対を生成（$H$ + CNOT）
2. **Alice の操作:** $q_0, q_1$ に CNOT → $q_0$ に $H$ → $q_0, q_1$ を測定
3. **Bob の補正:** 測定結果に応じて $X^{m_1}$, $Z^{m_0}$ を $q_2$ に適用

ノートでは $q_1, q_2, q_3$ と番号を振っているが、Qiskit では 0 始まりなので $q_0, q_1, q_2$ に対応させる：

| ノートの記号 | Qiskit | 所持者 | 役割 |
|---|---|---|---|
| $q_1$ | `q[0]` | Alice | 転送したい状態 $|\psi\rangle$ |
| $q_2$ | `q[1]` | Alice | ベル対の Alice 側 |
| $q_3$ | `q[2]` | Bob | ベル対の Bob 側 |

In [ ]:
def teleportation_circuit(init_gate=None):
    """量子テレポーテーション回路を構築する。
    
    init_gate: q[0] に適用する初期化ゲート（省略時は |0⟩ のまま）
    """
    q = QuantumRegister(3, "q")
    c = ClassicalRegister(2, "m")  # Alice の測定結果
    qc = QuantumCircuit(q, c)

    # --- 転送したい状態の準備 ---
    if init_gate is not None:
        qc.append(init_gate, [q[0]])
    qc.barrier(label="init")

    # --- 準備フェーズ: ベル対生成 (q[1], q[2]) ---
    qc.h(q[1])
    qc.cx(q[1], q[2])
    qc.barrier(label="Bell pair")

    # --- Alice の操作 ---
    qc.cx(q[0], q[1])      # CNOT: q[0](制御), q[1](標的)
    qc.h(q[0])              # アダマール: q[0]
    qc.barrier(label="Alice")

    # --- 測定 ---
    qc.measure(q[0], c[0])  # m[0] = q[0] の測定結果
    qc.measure(q[1], c[1])  # m[1] = q[1] の測定結果
    qc.barrier(label="measure")

    # --- Bob の補正 ---
    with qc.if_test((c[1], 1)):  # m[1]=1 なら X を適用
        qc.x(q[2])
    with qc.if_test((c[0], 1)):  # m[0]=1 なら Z を適用
        qc.z(q[2])

    return qc

# 回路を描画
qc_teleport = teleportation_circuit()
qc_teleport.draw("mpl", fold=-1)

## 3. シミュレーションによる検証

Qiskit の `c_if` は条件付きゲートだが、状態ベクトルシミュレータでは測定後の条件分岐を直接扱えない。
そこで、測定結果の4パターンそれぞれについて手動で補正を適用し、Bob の状態が元の $|\psi\rangle$ と一致するか検証する。

### 3.1 検証方法

測定なし・補正なしの回路で $|\Psi_2\rangle$ を構築し、Alice の測定結果ごとに Bob の状態を取り出す。

In [ ]:
from qiskit.circuit.library import UGate

def verify_teleportation(alpha, beta, label=""):
    """テレポーテーションを状態ベクトルレベルで検証する。
    
    転送したい状態: α|0⟩ + β|1⟩
    """
    # 転送したい元の状態
    psi = np.array([alpha, beta], dtype=complex)
    psi = psi / np.linalg.norm(psi)  # 正規化
    
    # 測定・補正なしの回路を構築（|Ψ₂⟩ を得る）
    qc = QuantumCircuit(3)
    
    # q[0] を |ψ⟩ に初期化
    qc.initialize(psi, 0)
    
    # ベル対生成: q[1], q[2]
    qc.h(1)
    qc.cx(1, 2)
    
    # Alice の操作
    qc.cx(0, 1)
    qc.h(0)
    
    # 状態ベクトルを取得
    sv = Statevector.from_instruction(qc)
    amplitudes = np.array(sv)
    
    # 補正ゲート行列
    I_gate = np.eye(2)
    X_gate = np.array([[0, 1], [1, 0]])
    Z_gate = np.array([[1, 0], [0, -1]])
    
    # bits = "m0 m1" (m0: q[0]の測定結果, m1: q[1]の測定結果)
    correction_names = {"00": "I", "01": "X", "10": "Z", "11": "ZX"}
    corrections = {
        "00": I_gate,
        "01": X_gate,
        "10": Z_gate,
        "11": Z_gate @ X_gate,
    }
    
    print(f"{'='*50}")
    print(f"転送する状態: {psi[0]:.4f}|0⟩ + {psi[1]:.4f}|1⟩  {label}")
    print(f"{'='*50}")
    
    all_match = True
    for bits, correction in corrections.items():
        m0, m1 = int(bits[0]), int(bits[1])
        
        # Qiskit のリトルエンディアン: index = q0 + q1*2 + q2*4
        idx0 = m0 + m1 * 2 + 0 * 4  # q2=0
        idx1 = m0 + m1 * 2 + 1 * 4  # q2=1
        bob_state = np.array([amplitudes[idx0], amplitudes[idx1]])
        
        # 正規化（各測定結果の確率は 1/4）
        norm = np.linalg.norm(bob_state)
        if norm > 1e-10:
            bob_state = bob_state / norm
        
        # 補正を適用
        corrected = correction @ bob_state
        
        # グローバル位相を除いて比較
        if abs(corrected[0]) > 1e-10:
            phase = psi[0] / corrected[0]
        elif abs(corrected[1]) > 1e-10:
            phase = psi[1] / corrected[1]
        else:
            phase = 1
        corrected_aligned = corrected * phase
        
        match = np.allclose(corrected_aligned, psi, atol=1e-6)
        all_match = all_match and match
        status = "✓" if match else "✗"
        
        print(f"\n  Alice の測定: |{bits}⟩")
        print(f"  Bob の状態（補正前）: {bob_state[0]:.4f}|0⟩ + {bob_state[1]:.4f}|1⟩")
        print(f"  補正ゲート: {correction_names[bits]}")
        print(f"  Bob の状態（補正後）: {corrected[0]:.4f}|0⟩ + {corrected[1]:.4f}|1⟩  {status}")
    
    print(f"\n  全パターンで |ψ⟩ を復元: {'成功' if all_match else '失敗'}")
    return all_match

### 3.2 具体例: $|\psi\rangle = |+\rangle$ のテレポーテーション

ノートの具体例と同じ $|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ を転送する。

In [ ]:
# |+⟩ = (|0⟩ + |1⟩) / √2
verify_teleportation(1/np.sqrt(2), 1/np.sqrt(2), label="(|+⟩)")

### 3.3 さまざまな状態で検証

$|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$, および一般的な状態で検証する。

In [ ]:
test_cases = [
    (1, 0, "|0⟩"),
    (0, 1, "|1⟩"),
    (1/np.sqrt(2), 1/np.sqrt(2), "|+⟩"),
    (1/np.sqrt(2), -1/np.sqrt(2), "|−⟩"),
    (np.cos(np.pi/8), np.sin(np.pi/8), "cos(π/8)|0⟩ + sin(π/8)|1⟩"),
    (1/np.sqrt(3), np.sqrt(2/3)*np.exp(1j*np.pi/4), "複素振幅の状態"),
]

results = []
for alpha, beta, label in test_cases:
    ok = verify_teleportation(alpha, beta, label=label)
    results.append((label, ok))
    print()

print("=" * 50)
print("まとめ:")
for label, ok in results:
    print(f"  {label}: {'成功' if ok else '失敗'}")

## 4. 古典通信なしの場合 — 完全混合状態の確認

ノートの「古典通信の必要性」セクションで示された通り、補正なしでは Bob の $q_2$ は完全混合状態 $\rho = I/2$ になることを確認する。

In [ ]:
from qiskit.quantum_info import partial_trace, DensityMatrix

def check_no_classical_communication(alpha, beta):
    """古典通信なしの場合、Bob の状態が完全混合状態になることを確認する。"""
    psi = np.array([alpha, beta], dtype=complex)
    psi = psi / np.linalg.norm(psi)
    
    # 測定なし・補正なしの回路
    qc = QuantumCircuit(3)
    qc.initialize(psi, 0)
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    
    # 全体の状態ベクトル → 密度行列
    sv = Statevector.from_instruction(qc)
    dm = DensityMatrix(sv)
    
    # q[0], q[1] をトレースアウトして Bob の q[2] の縮約密度行列を得る
    rho_bob = partial_trace(dm, [0, 1])
    
    print(f"転送する状態: {psi[0]:.4f}|0⟩ + {psi[1]:.4f}|1⟩")
    print(f"\nBob の縮約密度行列 ρ_bob:")
    print(np.array(rho_bob).round(6))
    print(f"\nI/2 との一致: {np.allclose(np.array(rho_bob), np.eye(2)/2, atol=1e-6)}")
    print(f"→ 古典通信なしでは |ψ⟩ の情報は一切得られない")

# 一般的な状態で確認
check_no_classical_communication(np.cos(np.pi/8), np.sin(np.pi/8) * np.exp(1j*np.pi/3))